# Feature Selection:

Now that we have analysed the data, we are ready to choose the features that will be used to train the model.

We first need to choose and preare the group of point-value metrics for the classical ML models.
Then we will need to compute the time-series features derived from the raw data like in 09.2_summarizing_group_of_gait_cycles.ipynb

Once selected, we will analyze the features in detail to choose the ones we will mvoe forward with to the next phase....


In [1]:
import core.constants as c
from core.utils import save_df_as_table_image
import os

import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, chi2_contingency, pearsonr
from sklearn.feature_selection import mutual_info_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import LabelEncoder

In [2]:
data = pd.read_csv(c.RICKD_FEATURE_SELECTION_DATA_FILE)
display(data.head())

data.info(verbose=True)

,id,speed_output,step_width_left,step_width_right,stride_rate_left,stride_rate_right,stride_length_left,stride_length_right,swing_time_left,swing_time_right,...,pelvic_drop_peak_vel_left,pelvic_drop_peak_vel_right,vertical_oscillation_left,vertical_oscillation_right,age,height,weight,gender,dominantleg,is_injured
0,100001_20110531T161051,2.489233,0.123419,0.123419,78.947368,78.947368,1.891817,1.891817,0.4350,0.420,...,-82.090017,-57.417845,96.433793,92.586134,47.0,172.0,61.9,female,left,True
1,100002_20110601T140505,2.722687,0.032922,0.032922,81.632653,81.632653,2.001175,2.001175,0.4450,0.420,...,-57.724685,-60.462411,86.521432,94.945518,37.0,173.4,70.6,male,left,True
2,100003_20110601T095930,2.949904,0.097273,0.097273,78.947368,78.947368,2.241927,2.241927,0.4475,0.445,...,-82.171073,-95.302664,82.680986,76.611379,51.0,186.0,86.5,male,right,True
3,100004_20110203T120721,2.688014,0.011401,0.011401,82.191781,82.191781,1.962250,1.962250,0.4400,0.460,...,-63.319119,-49.646132,92.593339,83.273183,35.0,175.6,59.0,male,left,True
4,100004_20140929T102035,2.928598,0.029021,0.029021,83.333333,82.758621,2.108590,2.123233,0.4200,0.430,...,-32.717949,-52.004473,87.481400,75.091789,39.0,175.0,61.0,male,left,False


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1441 entries, 0 to 1440
Data columns (total 88 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              1441 non-null   object 
 1   speed_output                    1441 non-null   float64
 2   step_width_left                 1441 non-null   float64
 3   step_width_right                1441 non-null   float64
 4   stride_rate_left                1441 non-null   float64
 5   stride_rate_right               1441 non-null   float64
 6   stride_length_left              1441 non-null   float64
 7   stride_length_right             1441 non-null   float64
 8   swing_time_left                 1441 non-null   float64
 9   swing_time_right                1441 non-null   float64
 10  stance_time_left                1441 non-null   float64
 11  stance_time_right               1441 non-null   float64
 12  pelvis_peak_drop_angle_left     14

In [3]:
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    dof = nx + ny - 2
    pooled_std = np.sqrt(((nx - 1) * np.var(x, ddof=1) + (ny - 1) * np.var(y, ddof=1)) / dof)
    return (np.mean(x) - np.mean(y)) / pooled_std

def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k - 1)*(r - 1))/(n - 1))
    rcorr = r - ((r - 1)**2)/(n - 1)
    kcorr = k - ((k - 1)**2)/(n - 1)
    return np.sqrt(phi2corr / min((kcorr - 1), (rcorr - 1)))

def calculate_vif(df):
    vif_data = pd.DataFrame()
    vif_data['feature'] = df.columns
    vif_data['VIF'] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
    return vif_data

# ==============================
# Main analysis function
# ==============================

def predictor_analysis(df, target, cat_threshold=10):
    results = []
    
    y = df[target]
    y_num = LabelEncoder().fit_transform(y)  # Binary 0/1 target
    
    for col in df.columns:
        if col == target:
            continue
        
        series = df[col].dropna()
        
        # Detect variable type
        if pd.api.types.is_numeric_dtype(series) and df[col].nunique() > cat_threshold:
            # Continuous variable
            group0 = df[df[target] == y.unique()[0]][col]
            group1 = df[df[target] == y.unique()[1]][col]
            
            # t-test
            t_pval = ttest_ind(group0, group1, equal_var=False).pvalue
            d = cohens_d(group0, group1)
            
            # Pearson
            r_val, r_pval = pearsonr(df[col], y_num)
            
            # Mutual Information
            mi_val = mutual_info_classif(df[[col]], y_num, discrete_features=False)[0]
            
            results.append({
                'predictor': col,
                'type': 'continuous',
                't-test p': t_pval,
                "Cohen's d": d,
                'Pearson r': r_val,
                'Pearson p': r_pval,
                'Mutual Info': mi_val
            })
        
        else:
            # Categorical variable
            contingency = pd.crosstab(df[col], y)
            chi2_pval = chi2_contingency(contingency)[1]
            cramer_v = cramers_v(contingency)
            mi_val = mutual_info_classif(pd.get_dummies(df[[col]]), y_num, discrete_features=True)[0]
            
            results.append({
                'predictor': col,
                'type': 'categorical',
                'Chi2 p': chi2_pval,
                "Cramer's V": cramer_v,
                'Mutual Info': mi_val
            })
    
    results_df = pd.DataFrame(results)
    
    # VIF for continuous predictors
    continuous_cols = [col for col in df.columns if col != target and pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() > cat_threshold]
    if continuous_cols:
        vif_df = calculate_vif(df[continuous_cols].fillna(0))
    else:
        vif_df = pd.DataFrame(columns=['feature', 'VIF'])
    
    return results_df, vif_df


In [4]:
excluded_columns = ["id"]
results_df, vif_df = predictor_analysis(data.drop(excluded_columns, axis=1), target='is_injured')

with pd.option_context('display.max_rows', None):
    display(results_df.sort_values('Mutual Info', ascending=False))
    display(vif_df.sort_values('VIF', ascending=False))

save_df_as_table_image(results_df, os.path.join(c.RICKD_RESULTS_FOLDER, 'predictor_analysis_results.png'))
save_df_as_table_image(vif_df, os.path.join(c.RICKD_RESULTS_FOLDER, 'predictor_analysis_VIF.png'))

/Users/adrianzapaterreig/Documents/Personal/TFM/rickd-analysis/.venv/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,predictor,type,t-test p,Cohen's d,Pearson r,Pearson p,Mutual Info,Chi2 p,Cramer's V
27,knee_flex_peak_angle_left,continuous,0.002728,-0.173161,-0.077065,0.003420,0.037381,NaN,NaN
83,weight,continuous,0.435431,0.044333,0.019785,0.452968,0.031274,NaN,NaN
54,foot_ang_at_hs_right,continuous,0.096196,0.099508,0.044374,0.092213,0.020574,NaN,NaN
75,hip_add_peak_vel_left,continuous,0.437795,-0.045875,-0.020473,0.437408,0.019435,NaN,NaN
65,hip_abd_peak_vel_left,continuous,0.457402,-0.043910,-0.019597,0.457283,0.019143,NaN,NaN
44,hip_add_peak_angle_right,continuous,0.093254,0.093717,0.041797,0.112754,0.018823,NaN,NaN
34,knee_abd_peak_angle_right,continuous,0.030757,-0.123134,-0.054881,0.037244,0.018244,NaN,NaN
0,speed_output,continuous,0.006008,-0.156037,-0.069483,0.008327,0.017508,NaN,NaN
67,knee_rot_peak_vel_left,continuous,0.978988,0.001598,0.000713,0.978418,0.017031,NaN,NaN
23,ankle_rot_peak_angle_left,continuous,0.640648,0.028601,0.012766,0.628242,0.016267,NaN,NaN


,feature,VIF
2,step_width_right,inf
1,step_width_left,inf
6,stride_length_right,2.817044e+06
5,stride_length_left,2.815172e+06
3,stride_rate_left,2.799797e+06
4,stride_rate_right,2.798121e+06
8,swing_time_right,1.669192e+04
7,swing_time_left,1.668037e+04
0,speed_output,1.145113e+04
10,stance_time_right,8.215843e+03
